# Dynamic impact — dropping a sphere onto a soft block

**The question.** Contact is rarely static. When a rigid sphere falls onto a soft block, how do Newton's fast solvers track the *transient* — impact, deformation, rebound — against an implicit FEM reference?

**The test.** A soft block rests on the ground; a **free** rigid sphere falls under gravity and strikes it. Newton runs all three solvers (XPBD / VBD / explicit) through its `soft_contact` pipeline; FEM solves implicit **Newmark-β** elastodynamics with penalty contact against the ground and the sphere, plus ~10% Kelvin–Voigt damping for impact stability.

**The three Newton solvers** (NVIDIA Newton is a GPU soft-body engine built on Warp/CUDA — same physics, different numerics):

* **XPBD** — *Extended Position-Based Dynamics*: fast positional projection (the game/robotics default). It satisfies constraints by moving positions, never solving a force balance, so it leaves a finite residual and reads soft.
* **VBD** — *Vertex Block Descent*: implicit; block coordinate descent on the backward-Euler objective over a coloured vertex graph. Converges toward the FEM-like solution as iterations grow — the apples-to-apples match for the implicit FEM.
* **SemiImplicit** — explicit, force-based (symplectic / semi-implicit Euler: a velocity-then-position step, no implicit solve). The one Warp can differentiate, used for the θ\* stiffness fit.

**Read this one carefully — it is the hardest, least-settled scenario.** The transient gap is *not* solver-only: the two sides also differ in material (Newton StVK/co-rotational vs FEM Neo-Hookean), contact model and time integration, and the material difference grows once impact strains leave the small-strain regime. So even an implicit **VBD** curve against Newmark is a *partial* fairness fix, not a clean solver-only comparison (see `docs/CONTACT.md`). The FEM drop's `dt`/damping are still being tuned, so its numbers are observations, not a settled benchmark; on the latest run only **VBD** makes a genuine two-way impact, while the **SemiImplicit** drop blows up.

> Whichever runs are present on disk are overlaid. The FEM drop data (`data/fem_drop.npz`) may be absent — then only the Newton solvers are shown.

In [ ]:
%matplotlib inline
import os

import matplotlib.pyplot as plt
import numpy as np

from common import params
from compare import drop as cmp_drop   # shared make_* helpers (single source for the figures)
from compare import style

# All present Newton drop runs, in canonical order/colour (style.load_newton_runs).
newtons = style.load_newton_runs(params.NEWTON_DROP_NPZ)
fem = np.load(params.FEM_DROP_NPZ) if os.path.exists(params.FEM_DROP_NPZ) else None
fem_hist = fem['history'] if fem is not None else None
if not newtons and fem_hist is None:
    raise FileNotFoundError('run newton_run.run_drop and/or fenics_run.run_drop first')
print('Newton drop runs:', [lbl for lbl, *_ in newtons] or 'none',
      '| FEM drop:', 'present' if fem_hist is not None else 'MISSING (run fenics_run.run_drop)')

## The scene — what we're actually simulating

A soft block resting on the ground (z = 0) with a rigid sphere dropped onto it. Below is the block at the **deepest genuine impact** — the run whose sphere sits lowest among those that actually impact and stay bounded (on this drop that is **VBD**; the canonical XPBD drop never reaches the block, and the explicit run blows up) — rendered from the real mesh, with the sphere as a wireframe at its lowest point.

In [ ]:
# Scene of the deepest impact (shared compare.drop -> same as the pipeline figure).
if newtons:
    nd, lbl = cmp_drop.scene_run(newtons, fem_hist)      # the run that indents deepest, not necessarily XPBD
    cmp_drop.make_scene(nd, lbl)
    plt.show()
else:
    print('No Newton drop mesh to render yet -- run: python -m newton_run.run_drop')

## 1. The verdict — the sphere's trajectory

The sphere's centre height over time: free fall, the impact knee, and rebound. Every present Newton solver is overlaid against the implicit Newmark FEM (when its data exists).

In [ ]:
cmp_drop.make_series(newtons, fem_hist, 1, "sphere centre height [m]",
                     "Drop: sphere trajectory (impact & rebound)")
plt.show()

## 2. Penetration — how far the sphere sinks into the block

Maximum sphere/block interpenetration. **Penetration is only a contact-quality measure where a genuine impact occurs:** a solver also reads ~0 mm when the sphere never reaches the block, or when the run goes unstable. The table below flags, per solver, whether the sphere actually descended to the block top (`z ≤ Lz + R`) and whether the strain energy stayed bounded; the curve then overlays only the genuine impacts. On this *free*-sphere drop only **VBD** drives the sphere down into a two-way contact — XPBD leaves it near its release height, and the explicit run goes unstable — so VBD is the meaningful Newton penetration to read against the implicit Newmark FEM.

In [ ]:
for r in cmp_drop.impact_table(newtons, fem_hist):
    if r['genuine']:
        note = 'genuine two-way impact'
    elif not r['reached']:
        note = 'sphere never reached the block -> pen=0 is trivial, not good contact'
    else:
        note = 'unstable / energy blow-up -> penetration reading not meaningful'
    print(f"{r['label']:14s} min z={r['min_z']:.3f} m  max pen={r['max_pen_mm']:6.2f} mm  "
          f"peak U={r['peak_U']:.4g} J  [{note}]")
cmp_drop.make_penetration(newtons, fem_hist)
plt.show()

## 3. Block strain energy

Elastic energy stored in the block as the impact deforms it. Each curve is the **same Neo-Hookean energy density** (the shared `compare/energies.py` diagnostic) evaluated on that solver's deformed configuration, so the energies are comparable even though the Newton solvers integrate an StVK/co-rotational law internally and use tet meshes while the FEM block is hex.

In [ ]:
cmp_drop.make_series(newtons, fem_hist, 3, "block strain energy [J]",
                     "Drop: block internal (strain) energy", logy=True)
plt.show()

## 4. Block kinetic energy

The block's kinetic energy through impact and rebound — the transient signature of how each solver moves energy through the collision.

In [ ]:
cmp_drop.make_series(newtons, fem_hist, 4, "block kinetic energy [J]",
                     "Drop: block kinetic energy", logy=True)
plt.show()

## 5. Contact force — FEM only

FEM assembles the sphere/block contact force directly. Newton's fast soft-contact path reports kinematics (trajectory, penetration, energies) but no calibrated contact force, so this axis is FEM-only — and needs the FEM drop run (`fenics_run.run_drop`) to have produced `data/fem_drop.npz`.

In [ ]:
fig = cmp_drop.make_contact_force(fem_hist)
if fig is not None:
    plt.show()
else:
    print("FEM drop contact force not available yet -- data/fem_drop.npz is absent.")
    print("Run fenics_run.run_drop (it needs the numpy/UFL fix) to produce it; the Newton runs")
    print("report kinematics and energies, not a calibrated contact force.")

## Takeaway

* The drop is the **dynamic counterpart** to the static indentation (`20_contact`): a *free* sphere makes the implicit **VBD** the natural — and hardest — match for implicit **Newmark** FEM.
* The solvers are compared on the shared kinematics — **trajectory** (§1), **penetration** (§2) and **energies** (§3–§4); the **contact force** (§5) is FEM-only, exactly as in the static contact case.
* This transient mixes material, contact-model and time-integration differences on top of the solver, so treat the gap as **illustrative, not a clean solver-only measurement** — and the FEM drop itself is still being settled (its dt/damping are being tuned), while the **SemiImplicit** drop blows up at this budget.